In [ ]:
import sys
sys.path.append('../src')

from credit_risk.data.loader import load_german_credit
from credit_risk.models.baseline import build_baseline_pipeline, evaluate_model, build_lightgbm_pipeline
from credit_risk.features.prepare import split_features_target
from sklearn.model_selection import train_test_split
import shap

In [9]:
df = load_german_credit('../data/raw/german_credit.csv')

In [10]:
x, y = split_features_target(df)
X_train, X_test, Y_train, Y_test = train_test_split(
x, y, test_size=0.2, stratify=y, random_state=42
)  # ИЗМЕНЕНО: раньше вызывалось без аргументов — теперь передаём x, y и параметры разбиения

num_cols = X_train.select_dtypes(include='number').columns.tolist()  # ИЗМЕНЕНО: было x.select_dtypes(...) — теперь именно X_train, а не весь x
cat_cols = X_train.select_dtypes(include='object').columns.tolist()  # аналогично

Построй build_lightgbm_pipeline, обучи на train
Создай TreeExplainer — но есть нюанс: SHAP обычно работает после препроцессинга (на данных, которые уже прошли через ColumnTransformer), а не на сырых X. Значит понадобится либо доставать модель из pipeline отдельно (pipeline.named_steps['model']), либо трансформировать X_test через pipeline.named_steps['preprocessor'] перед тем как передавать в SHAP.

In [ ]:
pipeline = build_lightgbm_pipeline(num_cols, cat_cols)
pipeline.fit(X_train, Y_train)  # ИЗМЕНЕНО: убрано лишнее присваивание "model =" — fit меняет pipeline на месте

In [ ]:
model = pipeline.named_steps['model']
X_test_transformed = pipeline.named_steps['preprocessor'].transform(X_test)

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_transformed)
shap.summary_plot(shap_values, X_test_transformed)